# 05 — Evaluation Pipeline & Regression Testing

**Needs `GROQ_API_KEY`.** (Free — see the folder README.)

---

## From scattered metrics to a suite you can trust

You now have all the pieces:
- string metrics (nb 01)
- LLM-as-a-judge (nb 02)
- tracing for latency & cost (nb 03)
- RAG / agent metrics (nb 04)

This notebook assembles them into **one repeatable evaluation suite** — the kind
you'd actually wire into CI. The workflow every serious team uses:

```
golden dataset → run system → score with all metrics → report → GATE (pass/fail)
```

And the question this finally lets you answer with confidence:

> *"I changed the prompt. Is the agent better or worse — and did I break anything?"*

### What you'll build
1. A **golden dataset** (the fixed test set every version faces)
2. A **swappable agent** with two versions (A vs B)
3. A **metric registry** combining metrics from nb 01–02
4. An **eval runner** that scores + times every case
5. **A/B comparison** of the two versions
6. A **regression gate** that fails when quality drops below a saved baseline

## Step 1 — Setup

In [ ]:
import os, re, json, time
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file (see the README)"

from langchain_groq import ChatGroq
judge = ChatGroq(model="llama3-8b-8192", temperature=0)  # the grader

def safe_parse(text: str) -> dict:
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return {"_raw": text, "_parse_error": True}

print("Judge ready:", judge.model_name)

## Step 2 — The golden dataset

A **golden dataset** is the fixed, version-controlled set of cases you test *every*
build against. It's the contract: 'these are the things our agent must get right.'
You grow it over time — every bug you find becomes a new case so it can never
silently come back.

Each case has an `input`, the `expected` answer, and which metrics apply.

In [ ]:
GOLDEN_DATASET = [
    {"id": 1, "input": "What is the capital of Japan?",          "expected": "Tokyo"},
    {"id": 2, "input": "What is 15 + 27?",                       "expected": "42"},
    {"id": 3, "input": "Who developed the theory of relativity?", "expected": "Albert Einstein"},
    {"id": 4, "input": "What is the chemical symbol for gold?",    "expected": "Au"},
    {"id": 5, "input": "In one word, what gas do plants absorb?",  "expected": "Carbon dioxide"},
]
print(f"Golden dataset: {len(GOLDEN_DATASET)} cases")

## Step 3 — The system under test (two versions)

We evaluate a tiny LLM agent whose behavior is controlled by its **system prompt**.
This is the realistic scenario: you tweak the prompt and want to know if it helped.

- **Version A** ('terse') — a plain, minimal prompt.
- **Version B** ('precise') — a prompt that demands a short, exact answer.

Same dataset, same metrics, two prompts. Let the eval decide the winner.

In [ ]:
SYSTEM_PROMPTS = {
    "A_terse":   "Answer the question.",
    "B_precise": "Answer with ONLY the shortest exact answer — no full sentences, no extra words.",
}

def make_agent(version: str):
    """Return an agent function that uses the given system prompt."""
    system = SYSTEM_PROMPTS[version]
    def agent(question: str) -> str:
        messages = [("system", system), ("human", question)]
        return judge.invoke(messages).content.strip()
    return agent

# Peek at how each version answers the same question
q = "What is the capital of Japan?"
print("A_terse  ->", make_agent("A_terse")(q))
print("B_precise->", make_agent("B_precise")(q))

## Step 4 — The metric registry

A **registry** is just a dict of `name -> scoring function`, so the runner can
apply any set of metrics without knowing what they are. We bring back two from
earlier notebooks: a fast string metric (`contains`, nb 01) and an LLM judge
(`llm_correct`, nb 02). Mixing a cheap deterministic check with an expensive smart
one is the standard real-world setup.

In [ ]:
# --- from nb 01: cheap, deterministic ---
def normalize(t): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", "", t.lower())).strip()
def contains(pred, exp): return 1.0 if normalize(exp) in normalize(pred) else 0.0

# --- from nb 02: smart, LLM-judged (0/1) ---
JUDGE_PROMPT = """Is the STUDENT ANSWER correct given the REFERENCE? Meaning matters, not wording.
QUESTION: {q}
REFERENCE: {ref}
STUDENT ANSWER: {ans}
Respond ONLY as JSON: {{"correct": true or false}}"""

def llm_correct(question, pred, exp):
    out = safe_parse(judge.invoke(JUDGE_PROMPT.format(q=question, ref=exp, ans=pred)).content)
    return 1.0 if out.get("correct") is True else 0.0

# The registry. Note: contains takes (pred, exp); llm_correct also needs the question.
METRICS = {
    "contains":    lambda case, pred: contains(pred, case["expected"]),
    "llm_correct": lambda case, pred: llm_correct(case["input"], pred, case["expected"]),
}
print("Registered metrics:", list(METRICS))

## Step 5 — The eval runner

The engine. For each case it runs the agent (timing the call, à la nb 03), applies
every registered metric, and records a row. Same shape as nb 01's harness — now
with latency baked in and metrics that can see the whole case.

In [ ]:
def run_eval(agent, dataset, metrics):
    rows = []
    for case in dataset:
        start = time.perf_counter()
        prediction = agent(case["input"])
        latency_ms = (time.perf_counter() - start) * 1000
        scores = {name: fn(case, prediction) for name, fn in metrics.items()}
        rows.append({
            "id": case["id"], "input": case["input"], "expected": case["expected"],
            "prediction": prediction, "latency_ms": round(latency_ms, 1), **scores,
        })
    return rows

def aggregate(rows, metrics):
    n = len(rows)
    report = {name: round(sum(r[name] for r in rows) / n, 3) for name in metrics}
    lat = sorted(r["latency_ms"] for r in rows)
    report["avg_latency_ms"] = round(sum(lat) / n, 1)
    report["p95_latency_ms"] = lat[min(int(0.95 * n), n - 1)]   # tail latency users feel
    # a simple overall 'pass rate': a case passes if the LLM judge says correct
    report["pass_rate"] = round(sum(r["llm_correct"] for r in rows) / n, 3)
    return report

print("Runner defined.")

## Step 6 — Run version A

Let's evaluate the terse prompt against the full golden dataset and read its
report card.

In [ ]:
rows_a = run_eval(make_agent("A_terse"), GOLDEN_DATASET, METRICS)
report_a = aggregate(rows_a, METRICS)

print("VERSION A (terse) — per case")
for r in rows_a:
    print(f"  [{r['id']}] judge={r['llm_correct']:.0f} contains={r['contains']:.0f} "
          f"{r['latency_ms']:>6} ms  {r['prediction'][:40]!r}")
print("\nREPORT A:", report_a)

## Step 7 — A/B comparison: version A vs version B

Now the question that justifies this whole folder. Run **both** versions through
the *identical* dataset and metrics, then diff the reports. This is how you make
prompt/model decisions with evidence instead of vibes.

In [ ]:
rows_b = run_eval(make_agent("B_precise"), GOLDEN_DATASET, METRICS)
report_b = aggregate(rows_b, METRICS)

print(f"{'metric':<16} {'A_terse':>10} {'B_precise':>10}  {'winner':>8}")
print('-' * 50)
for key in report_a:
    a, b = report_a[key], report_b[key]
    # for latency, lower is better; for scores, higher is better
    lower_better = "latency" in key
    if a == b:        win = "tie"
    elif lower_better: win = "A" if a < b else "B"
    else:             win = "A" if a > b else "B"
    print(f"{key:<16} {a:>10} {b:>10}  {win:>8}")

### Reading the A/B table

You'll typically see B_precise win on `contains` (its terse answers embed the exact
expected string more often) while both score similarly on `llm_correct` (the judge
forgives phrasing). Latency is usually a wash. The decision isn't 'which is
perfect' — it's 'which is *better on the metrics we care about*', backed by numbers
you can show your team.

## Step 8 — The regression gate

This is what turns evaluation into **protection**. You save a **baseline** report
(your current production quality). On every change, you re-run the suite and
**fail loudly** if any metric drops below baseline (minus a small tolerance for
LLM noise). Drop this into CI and a quality regression can't reach users.

In [ ]:
BASELINE_FILE = "baseline.json"
TOLERANCE = 0.05   # allow tiny dips for judge non-determinism

def save_baseline(report, path=BASELINE_FILE):
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"Baseline saved to {path}")

def regression_gate(report, path=BASELINE_FILE, tolerance=TOLERANCE):
    """Return (passed, failures). Higher-is-better metrics only."""
    if not os.path.exists(path):
        save_baseline(report)
        print("No baseline existed — saved current report as the baseline.")
        return True, []
    baseline = json.load(open(path))
    failures = []
    for metric in ["pass_rate", "contains", "llm_correct"]:
        if metric in baseline and metric in report:
            if report[metric] < baseline[metric] - tolerance:
                failures.append(f"{metric}: {report[metric]} < baseline {baseline[metric]}")
    return len(failures) == 0, failures

# Treat version B as our champion and save it as the baseline
save_baseline(report_b)

## Step 9 — Simulate a regression and watch the gate fire

Let's prove the gate works. We create a deliberately **broken** version C (it
sometimes returns an empty answer) and run it through the gate against B's
baseline. A good gate turns this silent quality drop into a loud, blocking
**FAIL** — exactly what you want before a release.

In [ ]:
import random
random.seed(0)

def broken_agent(question: str) -> str:
    # 40% of the time it 'breaks' and returns nothing useful
    if random.random() < 0.4:
        return "..."
    return make_agent("B_precise")(question)

rows_c = run_eval(broken_agent, GOLDEN_DATASET, METRICS)
report_c = aggregate(rows_c, METRICS)
print("REGRESSED REPORT C:", report_c)

passed, failures = regression_gate(report_c)
print("\n" + ("✅ GATE PASSED" if passed else "❌ GATE FAILED — blocking release"))
for f in failures:
    print("   -", f)

## Step 10 — Save a human-readable report

Numbers in a notebook vanish. Real suites write a report to disk (JSON for
machines, Markdown for humans) so results are diffable in PRs and reviewable by
teammates. Here's a minimal Markdown report writer.

In [ ]:
def write_markdown_report(reports: dict, path="eval_report.md"):
    """reports: {version_name: report_dict}"""
    lines = ["# Evaluation Report", ""]
    metrics = list(next(iter(reports.values())).keys())
    header = "| metric | " + " | ".join(reports) + " |"
    sep = "|" + "---|" * (len(reports) + 1)
    lines += [header, sep]
    for m in metrics:
        row = f"| {m} | " + " | ".join(str(reports[v][m]) for v in reports) + " |"
        lines.append(row)
    open(path, "w").write("\n".join(lines))
    print(f"Wrote {path}\n")
    print("\n".join(lines))

write_markdown_report({"A_terse": report_a, "B_precise": report_b, "C_broken": report_c})

## Step 11 — Try it yourself

1. **Grow the golden set.** Add a tricky case (a question your agent gets wrong),
   re-run, and watch the pass rate reflect it. *Every bug becomes a permanent test.*
2. **Tune the gate.** Lower `TOLERANCE` to `0.0` and re-run Step 9 — the gate gets
   stricter. Too strict and LLM noise causes false alarms; too loose and real
   regressions slip through. Finding that balance is the real skill.
3. **Add a metric.** Register a latency budget metric (`1.0` if under 2000 ms else
   `0.0`) in `METRICS` and include it in the gate.

In [ ]:
# Your playground — add a latency-budget metric and re-evaluate version B.
def under_budget(case, pred, budget_ms=2000):
    return 1.0   # placeholder — wire real latency in via the runner if you like

my_metrics = {**METRICS}  # add your metric here
rows = run_eval(make_agent("B_precise"), GOLDEN_DATASET, my_metrics)
print(aggregate(rows, my_metrics))

## Summary

| What you built | What it taught |
|----------------|----------------|
| `GOLDEN_DATASET` | The fixed contract every version is tested against |
| `make_agent(version)` | A swappable system under test for A/B testing |
| `METRICS` registry | Mix cheap string checks with smart LLM judges |
| `run_eval` + `aggregate` | Run → score → time → report card (with p95 latency) |
| A/B comparison | Choose prompts/models with evidence, not vibes |
| `regression_gate` | Fail the build when quality drops below baseline |
| Markdown report | Results you can diff in a PR and share |

**The big takeaway:** evaluation only protects you when it's **repeatable and
gated**. A golden dataset + a metric registry + a regression gate is the
difference between *hoping* your change helped and *knowing* it did.

---

## 🎓 You finished the series!

Across five notebooks you went from `exact_match` to a CI-grade eval suite:

| # | You learned to... |
|---|-------------------|
| 01 | Measure outputs with metrics you wrote by hand |
| 02 | Grade open-ended answers with an LLM judge |
| 03 | See inside an agent with traces, tokens & cost |
| 04 | Score retrieval, faithfulness, relevance & tool calls |
| 05 | Tie it all into an A/B suite with a regression gate |

### Where to go next
- Swap your hand-built metrics for **`ragas` / `deepeval` / `promptfoo`** now that
  you know what they compute under the hood.
- Turn on **LangSmith / Langfuse** for production tracing (nb 03, Step 10).
- Move from **offline** eval (fixed dataset) to **online** eval (scoring live traffic).
- Promote your best eval criteria into **runtime guardrails** that block bad outputs.

Happy evaluating — now you can *prove* your agents are good. 🚀